# D - Dependency Inversion (Inversión de Dependencias)
Contexto

Cuando se anuncia un torneo en la página web es necesario enviar notificaciones a los jugadores.

Antes de aplicar DIP

In [ ]:
# Violación: GestorTorneos depende de una implementación concreta (DiscordNotifier)
class DiscordNotifier:
    def __init__(self, channel_id: str = '#general'):
        self.channel_id = channel_id
        self.last_sent = None
    def enviar(self, mensaje):
        print(f"Discord: {mensaje}")

class GestorTorneos:
    def __init__(self):
        self.notificador = DiscordNotifier()
    def anunciar_torneo(self, nombre):
        mensaje = f"El torneo {nombre} ha sido creado"
        self.notificador.enviar(mensaje)

gestor = GestorTorneos()
gestor.anunciar_torneo('League Of Legends World Cup')


¿Por qué viola DIP?

GestorTorneos depende directamente de DiscordNotifier. Si mañana la empresa decide usar correo electrónico, WhatsApp o notificaciones push, será necesario modificar el código de la clase principal.

Aplicando DIP - versión ampliada

In [ ]:
from abc import ABC, abstractmethod
from typing import List

class Notificador(ABC):
    def __init__(self):
        self.last = None
        self.channel = None
    @abstractmethod
    def enviar(self, mensaje):
        pass
    @abstractmethod
    def test_connection(self) -> bool:
        pass

class DiscordNotifier(Notificador):
    def __init__(self, channel_id: str = '#general'):
        super().__init__()
        self.channel = channel_id
        self.api_token = 'discord-token'
    def enviar(self, mensaje):
        self.last = mensaje
        print(f"Discord: {mensaje}")
    def test_connection(self) -> bool:
        return True

class EmailNotifier(Notificador):
    def __init__(self, smtp_server: str = 'smtp.example'):
        super().__init__()
        self.smtp_server = smtp_server
        self.from_addr = 'noreply@example.com'
    def enviar(self, mensaje):
        self.last = mensaje
        print(f"Email: {mensaje}")
    def test_connection(self) -> bool:
        return True

class PushNotifier(Notificador):
    def __init__(self, api_key: str = 'push-key'):
        super().__init__()
        self.api_key = api_key
    def enviar(self, mensaje):
        self.last = mensaje
        print(f"Push: {mensaje}")
    def test_connection(self) -> bool:
        return True

class GestorTorneos:
    def __init__(self, notificador: Notificador):
        self.notificador = notificador
        self.torneos: List[str] = []
    def anunciar_torneo(self, nombre):
        self.torneos.append(nombre)
        mensaje = f"El torneo {nombre} ha sido creado"
        self.notificador.enviar(mensaje)
    def listar_torneos(self) -> List[str]:
        return list(self.torneos)
    def cancelar_torneo(self, nombre):
        if nombre in self.torneos:
            self.torneos.remove(nombre)
            return True
        return False

# uso con distintas implementaciones
discord = DiscordNotifier()
email = EmailNotifier()
push = PushNotifier()

gestor_discord = GestorTorneos(discord)
gestor_email = GestorTorneos(email)
gestor_push = GestorTorneos(push)

gestor_discord.anunciar_torneo('Valorant Masters')
gestor_email.anunciar_torneo('Counter Strike Pro League')
gestor_push.anunciar_torneo('FIFA Champions Cup')
print('Torneos discord:', gestor_discord.listar_torneos())
gestor_discord.cancelar_torneo('Valorant Masters')
print('Torneos after cancel:', gestor_discord.listar_torneos())
